# Train a Char-Level GPT on TinyShakespeare

This notebook assembles all Phase 5 components into a complete training pipeline:

```
TinyShakespeare.txt
    → [CharLevelDataset]
    → [GPT] → logits
    → [CrossEntropyLoss] → loss
    → [AdamW optimizer] → gradient update
    → [repeat] → trained model → generate text!
```

### Key Concept: Label Shift

For a sequence of tokens $x_1, x_2, \dots, x_T$, the model predicts
$\hat{x}_2, \hat{x}_3, \dots, \hat{x}_{T+1}$ — the **next token** at each position.

The `CharLevelDataset` you built already handles this: `target = input shifted by 1`.

In [ ]:
import sys
from pathlib import Path

# Add project root so we can import core.*
project_root = str(Path.cwd().parent) if Path.cwd().name == "apps" else str(Path.cwd())
if project_root not in sys.path:
    sys.path.append(project_root)

import math

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from core.transformer import (
    GPT,
    CharTokenizer,
    create_dataloaders,
)
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────
# Model dimensions
VOCAB_SIZE = 65  # TinyShakespeare unique chars (auto-detected from data)
D_MODEL = 128  # Embedding / transformer dimension
N_HEADS = 8  # Number of attention heads
N_LAYERS = 4  # Number of GPTBlocks
D_FF = 512  # FeedForward hidden dim (4 * D_MODEL)

# Training
BLOCK_SIZE = 128  # Context length (T)
BATCH_SIZE = 128  # Sequences per batch
MAX_STEPS = 5000  # Total training steps
LR = 3e-4  # Peak learning rate
DROPOUT = 0.2  # Dropout rate
WEIGHT_DECAY = 0.1  # AdamW weight decay
WARMUP_STEPS = 200  # Linear warmup steps
GRAD_CLIP = 1.0  # Max gradient norm
EVAL_INTERVAL = 500  # Steps between evaluations
GEN_INTERVAL = 500  # Steps between text generation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

---
## 1. Data Loading

Use the `create_dataloaders()` convenience function.

TinyShakespeare will be downloaded automatically if not cached.

In [ ]:
tokenizer = CharTokenizer()
VOCAB_SIZE = tokenizer.vocab_size

train_loader, val_loader, train_ds, val_ds = create_dataloaders(
    tokenizer=tokenizer,
    block_size=BLOCK_SIZE,
    batch_size=BATCH_SIZE,
    data_dir=str(Path(project_root) / "data"),
)

print(f"Corpus encoded length: {len(train_ds.data) + len(val_ds.data)} tokens")
print(f"Train samples: {len(train_ds)}  (= {len(train_loader)} batches/epoch)")
print(f"Val samples:   {len(val_ds)}  (= {len(val_loader)} batches)")
print()

x_batch, y_batch = next(iter(train_loader))
print(f"Batch x shape: {x_batch.shape}  (batch_size, seq_len)")
print(f"Batch y shape: {y_batch.shape}  (batch_size, seq_len)")

---
## 2. Model Initialisation

Create a mini GPT with the hyperparameters above. Since we're doing
character-level modeling, `vocab_size` is automatically detected from
the tokenizer (typically ~65 for TinyShakespeare).

In [ ]:
model = GPT(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    max_seq_len=BLOCK_SIZE,
    d_ff=D_FF,
    dropout=DROPOUT,
)
model = model.to(DEVICE)

print(model)

---
## 3. Optimizer & LR Schedule

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


def lr_lambda(step: int) -> float:
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

---
## 4. Training Loop

This is the core of the notebook. Each training step:

1. Get a batch `(x, y)` from the train DataLoader
2. Forward pass: `logits = model(x)`
3. Compute loss: `F.cross_entropy(logits.view(-1, V), y.view(-1))`
4. Backward pass: `loss.backward()`
5. Gradient clipping: `torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)`
6. Update: `optimizer.step()` + `scheduler.step()`
7. Log: print step, loss, perplexity, LR
8. Every EVAL_INTERVAL: compute validation loss
9. Every GEN_INTERVAL: generate a sample to see the model improving

In [ ]:
train_losses = []
val_losses = []
step_lrs = []
step = 0

model.train()

while step < MAX_STEPS:
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits = model(x)
        loss = F.cross_entropy(
            logits.view(-1, VOCAB_SIZE),
            y.view(-1),
            label_smoothing=0.1,
        )

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

        optimizer.step()
        scheduler.step()

        train_losses.append(loss.item())
        step_lrs.append(scheduler.get_last_lr()[0])

        if step % 100 == 0:
            ppl = math.exp(loss.item())
            print(
                f"Step {step:5d} | loss {loss.item():.4f} | ppl {ppl:.2f} | "
                f"lr {step_lrs[-1]:.2e}"
            )

        if step > 0 and step % EVAL_INTERVAL == 0:
            model.eval()
            val_loss = 0.0
            n = 0
            with torch.no_grad():
                for xv, yv in val_loader:
                    xv, yv = xv.to(DEVICE), yv.to(DEVICE)
                    logits = model(xv)
                    vloss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yv.view(-1))
                    val_loss += vloss.item() * len(xv)
                    n += len(xv)
            val_loss /= n
            val_losses.append(val_loss)
            print(f"  └─ Val loss {val_loss:.4f} | val ppl {math.exp(val_loss):.2f}")
            model.train()

        if step > 0 and step % GEN_INTERVAL == 0:
            model.eval()
            prompt_ids = torch.tensor(
                [tokenizer.encode("ROMEO:", add_special_tokens=False)],
                dtype=torch.long,
                device=DEVICE,
            )
            with torch.no_grad():
                output = model.generate(prompt_ids, max_new_tokens=80, temperature=0.8)
            sample_text = tokenizer.decode(output[0].tolist())
            print(f"  └─ Sample [{step}]: {sample_text[:120]}...")
            model.train()

        step += 1
        if step >= MAX_STEPS:
            break

print("Training complete!")

---
## 5. Training Curves

Plot the loss and perplexity over time to verify learning.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Training loss curve
ax = axes[0]
ax.plot(train_losses, alpha=0.6, label="train loss")
if val_losses:
    val_steps = list(range(0, len(train_losses), EVAL_INTERVAL))[: len(val_losses)]
    ax.plot(val_steps, val_losses, "o-", label="val loss", markersize=4)
ax.set_xlabel("Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Training & Validation Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Learning rate schedule
ax = axes[1]
ax.plot(step_lrs)
ax.set_xlabel("Step")
ax.set_ylabel("Learning Rate")
ax.set_title("LR Schedule (Warmup + Cosine)")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 6. Text Generation

Now that the model is trained, let's see what it learned!

Generate from the same prompt with different temperatures to see how
the output quality changes.

In [ ]:
# ── Compare sampling strategies ────────────────────────────────────────
model.eval()
prompt = "ROMEO:"
prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
prompt_tensor = torch.tensor([prompt_ids], dtype=torch.long).to(DEVICE)

temperatures = [0.0, 0.5, 0.8, 1.2]
names = ["Greedy (T=0.0)", "Conservative (T=0.5)", "Creative (T=0.8)", "Random (T=1.2)"]

for temp, name in zip(temperatures, names):
    with torch.no_grad():
        output_ids = model.generate(
            prompt_tensor,
            max_new_tokens=200,
            temperature=temp,
            top_k=40,
            top_p=0.9,
        )
    generated = tokenizer.decode(output_ids[0].tolist())
    print(f"═══ {name} ═══")
    print(generated)
    print()

---
## 7. Save Model Weights

Save the trained model so we can load it in analysis notebooks
without retraining from scratch. Also save the tokenizer vocab
so the analysis notebook knows the character mappings.

In [ ]:
save_dir = Path(project_root) / "models"
save_dir.mkdir(parents=True, exist_ok=True)

# Save model state_dict
model_path = save_dir / "gpt_tinyshakespeare.pt"
torch.save(model.state_dict(), model_path)
print(
    f"Model weights saved to {model_path} ({model_path.stat().st_size / 1024:.1f} KB)"
)

# Save tokenizer vocab
vocab_path = save_dir / "tokenizer_vocab.pt"
torch.save(tokenizer.vocab, vocab_path)
print(f"Tokenizer vocab saved to {vocab_path}")

config = {
    "d_model": D_MODEL,
    "n_layers": N_LAYERS,
    "n_heads": N_HEADS,
    "d_ff": D_FF,
    "block_size": BLOCK_SIZE,
    "vocab_size": VOCAB_SIZE,
    "max_steps": MAX_STEPS,
    "final_train_loss": train_losses[-1] if train_losses else None,
}
torch.save(config, save_dir / "training_config.pt")
print("Training config saved")